# Physical pin-shift review

Judge **date on scan** only (is the heading on the image right?).
Session numbers are **parsing artefacts** (same sitting / other ledger → other number,
same date) — ignore them; sync sittings later. Secret series stay out of this pipeline.

Edges in play: **DATED_AS** (heading on scan). Proposed **PINNED_TO:physical**
is inferred — not archival fact.

Deck: continuous stretch `3185` scans 95–110 (dated scans) + ambiguous dates +
a sample of far jumps (sampled by |Δ|≥4 under the hood; that delta is not shown).

Actions: **approve** / **reject** / **defer**. Free-text **notes** save with every
decision, or via **Save note** without changing the prior action.
Persists to `session_anchor_shift_approvals`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from IPython.display import HTML, clear_output, display
import ipywidgets as w

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sequence_graph.physical_pin_review import (
    build_review_deck,
    card_html,
    load_approvals,
    record_approval,
    save_approvals,
)

print("repo root:", ROOT)

In [ ]:
# Stretch defaults match the continuous proof (3185 scans 95–110).
DECK = build_review_deck(
    inventory="3185",
    scan_lo=95,
    scan_hi=110,
    include_ambiguous=True,
    include_far=True,
    far_limit=40,
)
APPROVALS = load_approvals()
print(
    f"deck={len(DECK)}  "
    f"kinds={ {k: sum(1 for c in DECK if c['card_kind']==k) for k in sorted({c['card_kind'] for c in DECK})} }  "
    f"prior_approvals={len(APPROVALS)}"
)

In [ ]:
kind_filter = w.Dropdown(
    options=["all", "dated_scan", "ambiguous", "far_shift", "undecided"],
    value="dated_scan",
    description="Filter",
)
idx = w.IntSlider(value=0, min=0, max=max(len(DECK) - 1, 0), description="Card")
note_box = w.Textarea(
    value="",
    placeholder="Free-text note (two dates on scan, Dominica/nihil, garbled heading, …)",
    layout=w.Layout(width="100%", height="100px"),
    description="Note",
)
status = w.HTML(value="")
card_out = w.Output()


def _filtered() -> list:
    k = kind_filter.value
    if k == "all":
        return DECK
    if k == "undecided":
        return [c for c in DECK if c["review_id"] not in APPROVALS]
    return [c for c in DECK if c["card_kind"] == k]


def _sync_slider(*_):
    cards = _filtered()
    idx.max = max(len(cards) - 1, 0)
    if idx.value > idx.max:
        idx.value = idx.max
    _render()


def _current():
    cards = _filtered()
    if not cards:
        return None
    return cards[min(idx.value, len(cards) - 1)]


def _render(*_):
    card = _current()
    with card_out:
        clear_output(wait=True)
        if card is None:
            display(HTML("<p>No cards in this filter.</p>"))
            note_box.value = ""
            return
        prior = APPROVALS.get(card["review_id"])
        display(HTML(card_html(card, prior)))
        note_box.value = (prior or {}).get("note") or ""
        decided = sum(1 for c in DECK if c["review_id"] in APPROVALS)
        status.value = (
            f"<b>{idx.value + 1}/{len(_filtered())}</b> in filter · "
            f"{decided}/{len(DECK)} decided overall"
        )


def _decide(action: str):
    card = _current()
    if card is None:
        return
    record_approval(
        APPROVALS,
        card,
        action,
        note=note_box.value.strip(),
        selected_session=None,
    )
    save_approvals(APPROVALS)
    cards = _filtered()
    if action != "note" and idx.value < len(cards) - 1:
        idx.value += 1
    else:
        _render()


btn_approve = w.Button(description="Approve", button_style="success")
btn_reject = w.Button(description="Reject", button_style="danger")
btn_defer = w.Button(description="Defer", button_style="warning")
btn_note = w.Button(description="Save note", button_style="info")
btn_prev = w.Button(description="← Prev")
btn_next = w.Button(description="Next →")

btn_approve.on_click(lambda _: _decide("approve"))
btn_reject.on_click(lambda _: _decide("reject"))
btn_defer.on_click(lambda _: _decide("defer"))
btn_note.on_click(lambda _: _decide("note"))
btn_prev.on_click(lambda _: setattr(idx, "value", max(idx.value - 1, 0)) or _render())
btn_next.on_click(
    lambda _: setattr(idx, "value", min(idx.value + 1, idx.max)) or _render()
)

kind_filter.observe(_sync_slider, names="value")
idx.observe(_render, names="value")

ui = w.VBox([
    w.HBox([kind_filter, idx, status]),
    card_out,
    note_box,
    w.HBox([btn_approve, btn_reject, btn_defer, btn_note, btn_prev, btn_next]),
])
display(ui)
_sync_slider()

## Summary of saved approvals

Re-run after a review pass. Notes appear in the `note` column.

In [ ]:
import pandas as pd

rows = list(APPROVALS.values())
if not rows:
    print("No approvals yet.")
else:
    df = pd.DataFrame(rows)[
        [
            c
            for c in (
                "card_kind",
                "action",
                "dated_day",
                "inventory",
                "scan_num",
                "note",
                "decided_at",
            )
            if c in pd.DataFrame(rows).columns
        ]
    ]
    display(df.sort_values(["card_kind", "dated_day"], na_position="last"))
    print(df["action"].value_counts().to_dict())
    n_notes = (df["note"].fillna("").str.strip() != "").sum()
    print(f"cards with notes: {n_notes}/{len(df)}")